# 전방 거리 가져오기

In [1]:
! pip install rplidar-roboticia

Looking in indexes: https://pypi.org/simple, https://www.piwheels.org/simple


In [1]:
import time
import statistics
from rplidar import RPLidar
lidar = RPLidar('/dev/ttyUSB0', baudrate=460800)

try:
    front_window = []  # 정면 거리 버퍼
    last_print = time.time()

    for new_scan, quality, angle, distance in lidar.iter_measures():
        # --- 0도 ±5도 범위만 수집 ---
        if (angle <= 5 or angle >= 355) and distance > 0:
            front_window.append(distance)

        # --- 주기적으로 거리 출력 ---
        if time.time() - last_print > 0.2:  # 0.2초마다 출력
            if front_window:
                # 거리의 중앙값(median)을 사용하면 노이즈에 강함
                front_distance = statistics.median(front_window)
                print(f"Front distance: {front_distance:.1f} mm "
                      f"(samples={len(front_window)})")
                front_window.clear()
            last_print = time.time()

except KeyboardInterrupt:
    print("\nStopping lidar...")

finally:
    lidar.stop()
    lidar.stop_motor()
    lidar.disconnect()
    print("Lidar stopped and disconnected.")

Front distance: 78.0 mm (samples=1)
Front distance: 79.0 mm (samples=17)
Front distance: 78.8 mm (samples=22)
Front distance: 78.9 mm (samples=18)
Front distance: 80.0 mm (samples=19)
Front distance: 79.6 mm (samples=22)
Front distance: 80.6 mm (samples=22)
Front distance: 81.4 mm (samples=22)
Front distance: 79.8 mm (samples=28)
Front distance: 80.8 mm (samples=27)
Front distance: 80.8 mm (samples=25)
Front distance: 80.1 mm (samples=26)
Front distance: 80.1 mm (samples=26)
Front distance: 79.5 mm (samples=23)
Front distance: 80.9 mm (samples=22)
Front distance: 80.6 mm (samples=24)
Front distance: 80.5 mm (samples=23)
Front distance: 80.8 mm (samples=23)
Front distance: 81.0 mm (samples=22)
Front distance: 81.2 mm (samples=22)
Front distance: 80.8 mm (samples=25)

Stopping lidar...
Lidar stopped and disconnected.


# 전방, 후방, 오른쪽, 왼쪽 거리 가져오기

In [1]:
import time
import threading
import statistics
from rplidar import RPLidar
from IPython.display import clear_output  # ← 핵심 추가

PORT = '/dev/ttyUSB0'
lidar = RPLidar(PORT, baudrate=460800)

# --- 전역 변수 ---
front_dist = None
back_dist = None
left_dist = None
right_dist = None

ANGLE_WINDOW = 10
UPDATE_INTERVAL = 0.3

def in_angle_range(angle, center, window=ANGLE_WINDOW):
    # angle이 center ± window 범위에 포함되는지
    diff = abs((angle - center + 180) % 360 - 180)
    return diff <= window

def lidar_thread():
    # 라이다 데이터를 계속 읽어서 각 방향 거리 업데이트
    global front_dist, back_dist, left_dist, right_dist

    buf_front, buf_back, buf_left, buf_right = [], [], [], []
    last_update = time.time()

    for new_scan, quality, angle, distance in lidar.iter_measures():
        if distance <= 0:
            continue

        # 각 방향별 버퍼 추가
        if in_angle_range(angle, 0):
            buf_front.append(distance)
        elif in_angle_range(angle, 90):
            buf_right.append(distance)
        elif in_angle_range(angle, 180):
            buf_back.append(distance)
        elif in_angle_range(angle, 270):
            buf_left.append(distance)

        # 일정 주기마다 거리값 계산
        if time.time() - last_update > UPDATE_INTERVAL:
            if buf_front:
                front_dist = statistics.median(buf_front)
            if buf_back:
                back_dist = statistics.median(buf_back)
            if buf_left:
                left_dist = statistics.median(buf_left)
            if buf_right:
                right_dist = statistics.median(buf_right)
            buf_front.clear(); buf_back.clear()
            buf_left.clear(); buf_right.clear()
            last_update = time.time()

def monitor_thread():
    # 각 방향 거리 출력
    global front_dist, back_dist, left_dist, right_dist
    while True:
        time.sleep(0.3)
        clear_output(wait=True)  # 이전 출력 제거
        print("---- Distance Info ----")
        print(f"Front : {front_dist:.1f} mm" if front_dist else "Front : ---")
        print(f"Back  : {back_dist:.1f} mm" if back_dist else "Back  : ---")
        print(f"Right : {right_dist:.1f} mm" if right_dist else "Right : ---")
        print(f"Left  : {left_dist:.1f} mm" if left_dist else "Left  : ---")
        print("-----------------------")

print("Starting RPLidar...")
info = lidar.get_info()
print(info)

t1 = threading.Thread(target=lidar_thread, daemon=True)
t2 = threading.Thread(target=monitor_thread, daemon=True)
t1.start()
t2.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping lidar...")
finally:
    lidar.stop()
    lidar.stop_motor()
    lidar.disconnect()
    print("Lidar stopped and disconnected.")

---- Distance Info ----
Front : 79.5 mm
Back  : 406.8 mm
Right : 638.1 mm
Left  : 146.2 mm
-----------------------
Lidar stopped and disconnected.
